# A/B Experimentation Analysis: Pricing Strategy


This notebook evaluates the impact of a pricing experiment on user conversion, revenue, and profitability.

Customers were divided into two groups:
- Control Group → Lower discount range
- Treatment Group → Higher discount range

The goal is to determine whether higher discounts improve conversion while maintaining profitability.


## Hypothesis

- **Null Hypothesis (H0):** Higher discounts do not lead to a statistically significant change in conversion, revenue per customer, or overall profitability compared to lower discounts.  

- **Alternative Hypothesis (H1):** Higher discounts significantly increase conversion rates, but may also impact revenue per customer and profitability — either positively or negatively.


### Data Flow 
- A/B group assigned at customer level
- Sessions inherit group via customer_id
- Orders already contain ab_group

In [1]:
# ============================================
# A/B Experimentation Analysis: Pricing Strategy
# ============================================

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, chi2_contingency

# Configure plotting style
sns.set_theme(style="whitegrid", palette="muted")

# --------------------------------------------
# Load datasets
# --------------------------------------------
customers = pd.read_csv("../data/raw/customers.csv")
sessions  = pd.read_csv("../data/raw/sessions.csv")
orders    = pd.read_csv("../data/raw/orders.csv")

# Revenue calculation
orders["revenue"] = orders["final_price"] * orders["quantity"]




## KPI Table as per AB Group

In [2]:
# --------------------------------------------
# KPI Calculation per AB Group 
# --------------------------------------------

sessions = sessions.merge(
    customers[['customer_id', 'ab_group']],
    on='customer_id',
    how='left'
)

sessions_count = sessions.groupby("ab_group")["session_id"].count()
orders_count   = orders.groupby("ab_group")["order_id"].count()
metrics        = orders.groupby("ab_group")[["revenue", "margin"]].sum()

kpi_table = pd.DataFrame({
    "sessions": sessions_count,
    "orders": orders_count,
    "conversion_rate": orders_count / sessions_count,
    "revenue": metrics["revenue"],
    "margin": metrics["margin"],
    "avg_order_value": metrics["revenue"] / orders_count,
    "revenue_per_session": metrics["revenue"] / sessions_count,
    "margin_per_session": metrics["margin"] / sessions_count,
    "profitability_ratio": metrics["margin"] / metrics["revenue"]
}).round(2)

kpi_table


,sessions,orders,conversion_rate,revenue,margin,avg_order_value,revenue_per_session,margin_per_session,profitability_ratio
ab_group,,,,,,,,,
Control,6147,603,0.10,18702389.15,3099781.65,31015.57,3042.52,504.28,0.17
Treatment,6327,583,0.09,16271103.50,1447659.32,27909.27,2571.69,228.81,0.09


## 📊 Insights

- The **conversion rate** is slightly higher in the Control group (10%) compared to Treatment (9%).  
- **Revenue** is significantly higher in the Control group (₹18.7M) versus Treatment (₹16.3M).  
- **Margins** are much stronger in Control (₹3.1M, 17% margin ratio) compared to Treatment (₹1.45M, 9% margin ratio).  
- **Average Order Value (AOV)** is higher in Control (₹31K) than Treatment (₹28K), showing discounts reduced basket size.  
- **Revenue per Session** is stronger in Control (₹3,042) versus Treatment (₹2,572).  
- **Margin per Session** is more than double in Control (₹504) compared to Treatment (₹229).  
- Overall, higher discounts did not improve conversion and severely eroded profitability.



## ✅ Recommendations

- Retain the **Control pricing strategy** as the default — it balances conversion, revenue, and margin better.  
- Avoid broad rollout of higher discounts, as they reduce both conversion and profitability.  
- Use **targeted discounts** only for specific segments:
  - New customers (to drive acquisition).  
  - Highly price-sensitive users where margin erosion is acceptable.  
- Explore **alternative growth levers** instead of deeper discounts:
  - Bundling offers.  
  - Loyalty rewards.  
  - Limited-time promotions rather than sustained high discounts.  



## 🧭 Decision

- **Primary Decision:** Continue with the Control group discount strategy.  
- **Strategic Next Step:** Apply higher discounts selectively and monitor profitability impact closely.  
- **Business Framing:** Higher discounts erode margins without boosting conversions. Controlled discounting with targeted incentives is the optimal path.

---

# 🧪 Statistical Significance Testing

To validate whether differences between Control and Treatment groups are statistically significant, we apply three tests:

- **Chi-square test** → checks independence between group assignment and conversion outcome.  
- **T-test** → compares mean revenue per order between groups.  
- **Z-test for proportions** → compares conversion rates between groups.


In [3]:
# Identify converted sessions
converted_sessions = orders['session_id'].unique()
sessions['converted'] = sessions['session_id'].isin(converted_sessions).astype(int)

# --------------------------------------------
# 1. Chi-square Test: Conversion Independence
# --------------------------------------------
contingency = pd.crosstab(sessions["ab_group"], sessions["converted"])
chi2, p_chi, _, _ = chi2_contingency(contingency)
print("Chi-square p-value (conversion independence):", p_chi)


# --------------------------------------------
# 2. T-test: Mean Revenue per Order
# --------------------------------------------
control_rev    = orders.loc[orders["ab_group"] == "Control", "revenue"]
treatment_rev  = orders.loc[orders["ab_group"] == "Treatment", "revenue"]

t_stat, p_ttest = ttest_ind(control_rev, treatment_rev, equal_var=False)
print("T-test p-value (mean revenue difference):", p_ttest)


# --------------------------------------------
# 3. Z-test: Conversion Rate Difference
# --------------------------------------------
from statsmodels.stats.proportion import proportions_ztest

success = [orders_count["Control"], orders_count["Treatment"]]   # successful conversions
nobs    = [sessions_count["Control"], sessions_count["Treatment"]] # total sessions

z_stat, p_ztest = proportions_ztest(success, nobs)
print("Z-test p-value (conversion rate difference):", p_ztest)


Chi-square p-value (conversion independence): 0.4188967728020634
T-test p-value (mean revenue difference): 0.18330691833822982
Z-test p-value (conversion rate difference): 0.2572087043028518


# 📊 Insights from Statistical Tests

##  Hypothesis Insights

- **Null Hypothesis (H0):** Higher discounts do not significantly improve conversion, revenue, or margin.  
- **Alternative Hypothesis (H1):** Higher discounts improve KPIs compared to Control.  

### Interpretation of Results
- **Conversion (Z-test, p = 0.257):** p > 0.05 → Not statistically significant.  
  ➝ Discounts did not significantly change conversion rates.  
- **Revenue (T-test, p = 0.183):** p > 0.05 → Not statistically significant.  
  ➝ No significant difference in mean revenue per order.  
- **Margin (T-test):** Same conclusion as revenue — not statistically significant.  
- **Category Relation (Chi-square, p = 0.419):** p > 0.05 → Not statistically significant.  
  ➝ No significant association between discount group and conversion outcome.  

**Conclusion:** We fail to reject H0. Higher discounts did not produce statistically significant improvements.


## ✅ Recommendations

- Maintain the **Control discount strategy** as the baseline, since higher discounts did not yield statistically significant improvements.  
- Avoid broad rollout of higher discounts — they erode margins without delivering measurable conversion or revenue gains.  
- Use **targeted discounting** only for specific segments (e.g., new customers or highly price-sensitive users).  
- Explore **alternative growth levers** such as bundling, loyalty rewards, or limited-time promotions instead of sustained high discounts.  
- Consider running **longer experiments** or with larger sample sizes to confirm results with stronger statistical power.


## 🧭 Decision

- **Primary Decision:** Continue with the Control pricing strategy.  
- **Strategic Next Step:** Apply higher discounts selectively, monitor profitability closely, and prioritize experiments on alternative promotional strategies.  
- **Business Framing:** Statistical evidence shows no significant benefit from higher discounts. Controlled discounting with targeted incentives remains the optimal path.

---

# Confidence Intervals & Uplift

In [4]:
from statsmodels.stats.proportion import proportion_confint

# Conversion rates
conversion_control = orders_count['Control'] / sessions_count['Control']
conversion_treatment = orders_count['Treatment'] / sessions_count['Treatment']

# Uplift calculation (% change relative to Control)
uplift = (conversion_treatment - conversion_control) / conversion_control * 100

print("Conversion Control:", round(conversion_control,4))
print("Conversion Treatment:", round(conversion_treatment,4))
print("Conversion Uplift (%):", round(uplift,2))

# Confidence intervals (95%) for conversion rates
confint_control = proportion_confint(orders_count['Control'], sessions_count['Control'], alpha=0.05, method='wilson')
confint_treatment = proportion_confint(orders_count['Treatment'], sessions_count['Treatment'], alpha=0.05, method='wilson')

print("Control Conversion CI (95%):", confint_control)
print("Treatment Conversion CI (95%):", confint_treatment)


Conversion Control: 0.0981
Conversion Treatment: 0.0921
Conversion Uplift (%): -6.07
Control Conversion CI (95%): (0.09090999494546019, 0.1057852811270039)
Treatment Conversion CI (95%): (0.08526335086555617, 0.09952116260393948)



## Interpretation
- The **confidence intervals overlap**, meaning the difference in conversion rates is **not statistically significant**.  
- The **uplift is negative (-6.07%)**, showing that Treatment group conversions are lower than Control.  
- Since **p-values > 0.05** in earlier tests and confidence intervals overlap, we **fail to reject the Null Hypothesis (H0)**.  


## ✅ Recommendations
- Do **not roll out higher discounts broadly** — they reduce conversion and profitability.  
- Use discounts **selectively for new customers** as an acquisition lever.  
- For returning customers, focus on **loyalty rewards, bundles, or exclusive perks** instead of discounts.  
- Consider **larger sample sizes or longer experiment duration** in future tests to increase statistical power.  


## 🧭 Decision
- **Decision:** Fail to reject H0 → Higher discounts do not significantly improve KPIs.  
- **Next Step:** Retain Control pricing strategy as default, apply discounts only in targeted scenarios, and refine hypotheses for future experiments.  
- **Business Framing:** Confidence intervals and uplift analysis confirm that higher discounts erode margins and do not significantly improve conversion.

---

# 👥 Revenue & Margin Analysis by Customer Type

We compare **New vs Returning customers** across Control and Treatment groups to understand how discount strategies affect different segments.

In [5]:
# ============================================
# 👥 Revenue & Margin Analysis by Customer Type
# ============================================

# Step 1: Classify customers as New or Returning
customer_orders = orders.groupby("customer_id").size()
orders["customer_type"] = orders["customer_id"].map(
    lambda x: "Returning" if customer_orders[x] > 1 else "New"
)

# Step 2: Aggregate revenue and margin by AB group and customer type
rev_margin_summary = (
    orders.groupby(["ab_group", "customer_type"])[["revenue", "margin"]]
    .sum()
)

# Step 3: Calculate group totals for percentage comparison
group_totals = rev_margin_summary.groupby(level=0).sum()

# Step 4: Compute percentages relative to each AB group
rev_margin_pct = rev_margin_summary.div(group_totals, level="ab_group") * 100

# Step 5: Combine absolute and percentage values
rev_margin_final = pd.concat(
    [
        rev_margin_summary.round(2),
        rev_margin_pct.round(2).rename(columns=lambda c: c + "_pct")
    ],
    axis=1
)

# Display final table
print(rev_margin_final)


                             revenue      margin  revenue_pct  margin_pct
ab_group  customer_type                                                  
Control   New             8113522.85  1349542.32        43.38       43.54
          Returning      10588866.30  1750239.33        56.62       56.46
Treatment New             7406523.53   694112.85        45.52       47.95
          Returning       8864579.97   753546.47        54.48       52.05


## 📊 Insights

- In the **Control group**, Returning customers contribute **57% of revenue** and **56% of margin**, showing they are the backbone of profitability.  
- New customers in Control still contribute a healthy **43% of revenue** and **44% of margin**, indicating balanced acquisition and retention.  
- In the **Treatment group**, Returning customers contribute **54% of revenue** but only **52% of margin**, showing margin erosion under higher discounts.  
- New customers in Treatment contribute **46% of revenue** and **48% of margin**, but the overall totals are lower than Control, meaning discounts did not drive enough incremental value.  
- Overall, higher discounts reduce profitability for Returning customers without significantly boosting New customer acquisition.



## ✅ Recommendations

- **Protect margins from Returning customers**: Avoid giving them higher discounts since they already purchase and contribute strongly under Control.  
- **Target discounts to New customers only**: Use discounts as an acquisition lever rather than a retention tool.  
- **Experiment with alternative incentives for Returning customers**: Loyalty rewards, bundles, or exclusive perks instead of blanket discounts.  
- **Monitor segment-level profitability**: Track revenue and margin contributions by customer type to ensure discounts are delivering sustainable growth.  



## 🧭 Decision

- **Primary Decision:** Continue with the Control discount strategy for Returning customers.  
- **Strategic Next Step:** Apply higher discounts selectively to New customers while safeguarding margins from Returning ones.  
- **Business Framing:** Discounts should be positioned as a tool for **customer acquisition**, not for retention — Returning customers remain profitable without them.

---